In [1]:
!nvidia-smi

Tue Jul 14 16:52:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!curl -s ifconfig.me && echo " <- internet OK"

34.148.175.147 <- internet OK


In [3]:
import glob, os
MODEL_PATH = os.path.dirname(glob.glob("/kaggle/input/**/config.json", recursive=True)[0])
print("MODEL_PATH =", MODEL_PATH)
!du -sh {MODEL_PATH}

MODEL_PATH = /kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1
15G	/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1


In [4]:
!pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 97.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.4 MB/s eta 0:00:00


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_PATH)

has_cuda = torch.cuda.is_available()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16 if has_cuda else torch.float32,
    device_map="auto" if has_cuda else None,
)
model.eval()
print("loaded on:", model.device if hasattr(model, 'device') else next(model.parameters()).device)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

loaded on: cuda:0


In [7]:
# def search_law(query: str):
#     """Search Indian law texts and return the most relevant sections.

#     Args:
#         query: what to search for, e.g. 'security deposit refund tenancy'
#     """
#     return "stub"

# messages = [
#     {"role": "system", "content":
#      "You are a legal information assistant for citizens of India. "
#      "Use the available tools to look up the law before answering."},
#     {"role": "user", "content":
#      "My landlord has not returned my security deposit for 6 months. "
#      "Find the relevant law."},
# ]

# inputs = tok.apply_chat_template(
#     messages,
#     tools=[search_law],              # <- hands the tool schema to the template
#     add_generation_prompt=True,
#     return_dict=True, return_tensors="pt",
# ).to(model.device)

# out = model.generate(**inputs, max_new_tokens=200)
# print(tok.decode(out[0][inputs["input_ids"].shape[-1]:],
#                  skip_special_tokens=False))   # <- keep special tokens visible

In [8]:
import re

def gemma_chat(messages, tools=None, max_new_tokens=512):
    """One model turn. Returns (raw_with_special_tokens, clean_text)."""
    inputs = tok.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        stop_strings=["<tool_call|>"], tokenizer=tok)   # stop right after a call
    gen = out[0][inputs["input_ids"].shape[-1]:]
    return (tok.decode(gen, skip_special_tokens=False),
            tok.decode(gen, skip_special_tokens=True))

TOOL_RE = re.compile(r"<\|tool_call>call:(\w+)\{(.*?)\}(?:<tool_call\|>)?", re.S)
ARG_RE  = re.compile(r'(\w+)\s*:\s*<\|"\|>(.*?)<\|"\|>', re.S)

def parse_reply(raw):
    m = TOOL_RE.search(raw)
    if not m:
        return None                      # no tool call -> it's a normal answer
    name, blob = m.group(1), m.group(2)
    args = dict(ARG_RE.findall(blob))    # quoted string arguments
    for k, v in re.findall(r'(\w+)\s*:\s*([^,}<]+)', blob):
        args.setdefault(k, v.strip())    # unquoted args (numbers etc.)
    return {"name": name, "args": args}

def run_with_tools(messages, tools, max_steps=6):
    tool_map = {f.__name__: f for f in tools}
    messages = list(messages)
    seen = set()
    for _ in range(max_steps):
        raw, clean = gemma_chat(messages, tools=tools)
        call = parse_reply(raw)
        if call is None:
            return clean.strip(), messages
        print(f"[tool fired] {call['name']}({call['args']})")
        key = (call["name"], tuple(sorted(call["args"].items())))
        if key in seen:
            result = ("You already received these exact results. Do not search "
                      "again. Answer now from what you have, or state plainly "
                      "that the database does not cover this issue.")
        else:
            seen.add(key)
            result = tool_map[call["name"]](**call["args"])
        messages += [
            {"role": "assistant", "tool_calls": [{"type": "function",
                "function": {"name": call["name"], "arguments": call["args"]}}]},
            {"role": "tool", "name": call["name"], "content": str(result)},
        ]
    return "(loop limit reached)", messages

In [9]:
!pip -q install chromadb sentence-transformers pypdf
!pip -q install -U "opentelemetry-sdk>=1.43.0" "opentelemetry-api>=1.43.0" "opentelemetry-exporter-otlp-proto-common>=1.43.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 77.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [10]:
# Part C — the ingestion pipeline (dataset-agnostic: one PATH constant + a chunker
# that survives any document format via a fallback plan)
import re, glob, os
from pypdf import PdfReader

# Auto-detect the dataset folder rather than hardcoding it: Kaggle's mount path
# for datasets has changed before (flat /kaggle/input/<name> vs nested under
# /kaggle/input/datasets/...), so search for any folder containing PDFs instead
# of assuming a fixed layout.
_candidates = [d for d in glob.glob("/kaggle/input/**/practice-laws", recursive=True)
               if os.path.isdir(d)]
if not _candidates:
    # fall back to: any folder under /kaggle/input that directly contains a pdf
    _candidates = sorted({os.path.dirname(p) for p in
                           glob.glob("/kaggle/input/**/*.pdf", recursive=True)})

DOCS_PATH = _candidates[0] if _candidates else "/kaggle/input/practice-laws"
print("DOCS_PATH =", DOCS_PATH)
if len(_candidates) > 1:
    print("note: multiple candidate folders found, using the first:", _candidates)

def load_chunks(path):
    chunks = []
    pdfs = glob.glob(f"{path}/**/*.pdf", recursive=True)
    print("PDFs found:", pdfs)
    for pdf in pdfs:
        text = "\n".join((p.extract_text() or "") for p in PdfReader(pdf).pages)
        print(f"  {pdf} -> extracted {len(text)} chars")
        act = pdf.split("/")[-1].replace(".pdf", "")
        # Plan A: split on section numbering like "\n12. " (bare-act style)
        parts = re.split(r"\n(\d{1,3})\.\s", text)
        if len(parts) > 10:
            for i in range(1, len(parts) - 1, 2):
                body = parts[i + 1].strip()[:2000]
                if len(body) > 50:
                    chunks.append({"text": body, "act": act, "section": parts[i]})
        else:
            # Plan B: unknown format -> overlapping fixed windows
            for j in range(0, len(text), 1000):
                body = text[j:j + 1200].strip()
                if len(body) > 50:
                    chunks.append({"text": body, "act": act,
                                   "section": f"part-{j // 1000 + 1}"})
    return chunks

chunks = load_chunks(DOCS_PATH)
if not chunks:
    raise RuntimeError(
        "No chunks produced. Check the 'PDFs found' and 'extracted N chars' lines above: "
        "if PDFs found is empty, the dataset isn't attached/named right; if extracted "
        "chars is ~0, the PDF is scanned images with no text layer (grab a different "
        "act or the 'as passed' text version)."
    )
print(len(chunks), "chunks | sample:", chunks[0]["act"],
      "sec", chunks[0]["section"], "|", chunks[0]["text"][:120])

DOCS_PATH = /kaggle/input/datasets/kushkukreja/practice-laws
PDFs found: ['/kaggle/input/datasets/kushkukreja/practice-laws/eng201935.pdf']
  /kaggle/input/datasets/kushkukreja/practice-laws/eng201935.pdf -> extracted 134880 chars
145 chunks | sample: eng201935 sec 2 | Definitions. 
CHAPTER II 
CONSUMER PROTECTION COUNCILS.


In [11]:
# No embedding model needed — keyword search over the chunks
print("chunks ready:", len(chunks))

chunks ready: 145


In [12]:
# Part E — the real search_law, then the moment of truth.
# Redefine the tool (same name + docstring -> agent code needs zero changes)
# and re-run the exact agent loop from Step 3.
def search_law(query: str):
    """Search Indian law texts and return the most relevant sections.

    Args:
        query: what to search for, e.g. 'security deposit refund tenancy'
    """
    words = [w for w in re.findall(r"\w+", query.lower()) if len(w) > 2]
    scored = []
    for c in chunks:
        text_l = c["text"].lower()
        score = sum(text_l.count(w) for w in words)
        scored.append((score, c))
    scored.sort(key=lambda x: -x[0])
    top = [c for s, c in scored[:4] if s > 0] or [c for s, c in scored[:2]]
    return "\n\n".join(
        f"[{c['act']} — Section {c['section']}] {c['text']}" for c in top
    )

question = "I bought a defective phone online and the seller refuses a refund. What are my rights?"
messages = [
    {"role": "system", "content":
     "You are a legal information assistant for citizens of India. "
     "Always look up the law with the available tools before answering. "
     "Cite only acts and sections that appear in tool results."},
    {"role": "user", "content": question},
]
answer, _ = run_with_tools(messages, [search_law])
print(answer)


[tool fired] search_law({'query': 'consumer rights defective goods online purchase India'})
Based on the information available, your rights as a consumer in India regarding defective goods purchased online are protected under the Consumer Protection Act.

Here is a summary of what the law indicates:

1.  **Grounds for Complaint:** A complaint can be filed if the goods you bought "suffer from one or more defects" (as per **Section 2(6)(ii)**).
2.  **Remedies Available:** If a District Commission finds that the goods you complained about suffer from defects, it can issue an order directing the seller (the opposite party) to do one or more of the following (**Section 39(1)**):
    *   **Replace the goods** with new goods of similar description that are free from any defect (**Section 39(1)(b)**).
    *   **Return the price** you paid, along with any interest that may be decided (**Section 39(1)(c)**).
    *   **Remove the defect** pointed out by an appropriate laboratory (**Section 39(1)(

In [13]:
!pip -q install langgraph

In [14]:
# State schema shared across all four agent nodes
import json, re
from typing import TypedDict, List, Dict, Any

MAX_RETRIES = 2

class AgentState(TypedDict):
    question: str    
    language: str
    case_facts: Dict[str, Any]
    advisor_answer: str       # advisor's cited answer, pre-formatting
    retrieved_chunks: List[str]  # raw tool results collected from search_law calls
    draft: str                # drafter's case-card text
    eval_result: Dict[str, Any]
    retry_count: int
    needs_more_info: bool
    clarifying_questions: List[str]

def extract_json(text: str) -> dict:
    """Small local models don't always emit clean JSON -- grab the first {...} block."""
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {}

In [15]:
# Node 1 -- Interviewer: turns the raw question into structured case facts.
# One-shot version: asks the model to self-assess whether it has enough to proceed.
def interviewer_node(state: AgentState) -> AgentState:
    messages = [
        {"role": "system", "content":
         "You are the Interviewer stage of a legal-assistant pipeline for citizens "
         "of India. First detect the language the user wrote in. Respond with ONLY a "
         "JSON object, no other text, with fields: "
         '{"language": "English/Hindi/...", "ready": true/false, "issue_type": "...", '
         '"key_facts": "...", "missing_info": ["..."]}. '
         "Write the strings in missing_info IN THE USER'S LANGUAGE. "
         "Keep issue_type and key_facts in English for internal use. "
         "Set ready=false only if critical facts (what happened, roughly when, "
         "who the other party is) are genuinely missing."},
        {"role": "user", "content": state["question"]},
    ]
    _, clean = gemma_chat(messages, tools=None, max_new_tokens=250)
    facts = extract_json(clean)
    state["language"] = facts.get("language", "English") or "English"
    ready = bool(facts.get("ready", True))
    state["case_facts"] = facts
    state["needs_more_info"] = not ready
    state["clarifying_questions"] = facts.get("missing_info", [])
    return state

def route_after_interviewer(state: AgentState) -> str:
    return "end_needs_info" if state["needs_more_info"] else "advisor"

In [16]:
# Node 2 -- Advisor: does the actual retrieval + reasoning, via the existing
# run_with_tools loop and the Chroma-backed search_law tool. Also captures the
# raw tool results, so the evaluator has something concrete to check citations against.
def advisor_node(state: AgentState) -> AgentState:
    facts = state["case_facts"]
    revise_note = ""
    if state.get("eval_result", {}).get("issues"):
        revise_note = (
            " Your previous answer had these unverified citations, fix them or "
            "drop the claim: " + "; ".join(state["eval_result"]["issues"])
        )
    messages = [
        {"role": "system", "content":
         "You are a legal information assistant for citizens of India. "
         "Always look up the law with the available tools before answering. "
         "Cite only acts and sections that appear in tool results, in the exact "
         "'[Act — Section X]' format the tool returns. "
         "If the tool results do not actually cover the user's issue, do NOT name "
         "any act or law from memory. Say plainly that the available legal database "
         "does not cover this specific matter,In that case, begin your answer with "
         "the exact tag NO_COVERAGE on its own line, then give only general practical "
         "steps (written demand, keep records), and direct the user to the District"
         "Legal Services Authority (free legal aid) and helpline 15100."
         + revise_note},
        {"role": "user", "content":
         f"Case type: {facts.get('issue_type', '')}. "
         f"Facts: {facts.get('key_facts', state['question'])}"},
    ]
    answer, full_messages = run_with_tools(messages, [search_law])
    state["advisor_answer"] = answer
    state["retrieved_chunks"] = [
        m["content"] for m in full_messages if m.get("role") == "tool"
    ]
    return state

In [17]:
# Node 3 -- Drafter: pure formatting into a human-friendly case card.
# Explicitly told not to introduce new facts or citations.
def drafter_node(state: AgentState) -> AgentState:
    lang = state.get("language", "English")
    messages = [
        {"role": "system", "content":
         f"You are the Drafter stage. Reformat the Advisor's answer below into a "
         f"clean 'case card'. WRITE THE ENTIRE CARD IN {lang}, EXCEPT keep every "
         f"act name and every [Act — Section X] citation in English exactly as "
         f"written. Use these exact sections (translate the headings into {lang}): "
         f"Situation, Applicable Law, Recommended Steps, Deadlines (only if a time "
         f"limit appears in the Advisor's answer), Disclaimer (always include, in "
         f"{lang}: legal information not legal advice, consult a lawyer). "
         f"If the Advisor's answer contains the tag NO_COVERAGE, include that exact "
         f"tag NO_COVERAGE once at the very top of your card, before any translated text. "
         f"Do not add any fact, law, or citation not already in the Advisor's answer."},
        {"role": "user", "content": state["advisor_answer"]},
    ]
    _, clean = gemma_chat(messages, tools=None, max_new_tokens=450)
    state["draft"] = clean.strip()
    return state

In [18]:
# === THE ONE AND ONLY evaluator cell (delete all other evaluator/router cells) ===
import re

MAX_RETRIES = 2

def _norm(s):
    s = s.lower().replace("—", "-").replace("–", "-")
    return re.sub(r"\s+", " ", s).strip()

CITATION_RE = re.compile(r"\[([^\]]+?)\s*[—–-]\s*section\s*([^\]]+?)\]", re.I)
ACT_RE      = re.compile(r"\b((?:[A-Z][\w()&',-]*\s+){0,6}Act(?:,?\s*\d{4})?)")
NO_COVERAGE_MARKERS = ("does not cover", "does not contain", "database does not",
                       "not covered", "no specific provisions")

def evaluator_node(state: AgentState) -> AgentState:
    pool   = _norm("\n".join(state.get("retrieved_chunks", [])))
    draft  = state["draft"]
    d_norm = _norm(draft)
    issues = []

    bracketed = CITATION_RE.findall(draft)
    for act, section in bracketed:
        act_n, sec_n = _norm(act), _norm(section).rstrip(".")
        if act_n not in pool or f"section {sec_n}" not in pool:
            issues.append(f"unverified citation [{act.strip()} — Section {section.strip()}]")

    for m in ACT_RE.findall(draft):
        if _norm(m) not in pool:
            issues.append(f"'{m.strip()}' named but not in retrieved law")

    honest = "no_coverage" in d_norm or any(k in d_norm for k in NO_COVERAGE_MARKERS)

    if bracketed and not issues:
        passed, mode = True, "grounded"
    elif not bracketed and not issues and honest:
        passed, mode = True, "honest_no_coverage"
    else:
        passed, mode = False, "failed"
        if not bracketed and not honest:
            issues.append("no citations and no honest no-coverage statement")

    state["eval_result"] = {"passed": passed, "mode": mode, "issues": issues}
    if not passed:
        state["retry_count"] = state.get("retry_count", 0) + 1
    return state

def route_after_evaluator(state: AgentState) -> str:   # pure decider, no state changes
    if state["eval_result"]["passed"] or state["retry_count"] > MAX_RETRIES:
        return "end_done"
    return "advisor"

In [19]:
# Wire the graph together
from langgraph.graph import StateGraph, START, END

graph = StateGraph(AgentState)
graph.add_node("interviewer", interviewer_node)
graph.add_node("advisor", advisor_node)
graph.add_node("drafter", drafter_node)
graph.add_node("evaluator", evaluator_node)

graph.add_edge(START, "interviewer")
graph.add_conditional_edges(
    "interviewer", route_after_interviewer,
    {"advisor": "advisor", "end_needs_info": END},
)
graph.add_edge("advisor", "drafter")
graph.add_edge("drafter", "evaluator")
graph.add_conditional_edges(
    "evaluator", route_after_evaluator,
    {"advisor": "advisor", "end_done": END},
)

app = graph.compile()
print("graph compiled")

graph compiled


In [20]:
def draft_notice(case_facts: dict, language: str = "English") -> str:
    """Generate a formal legal demand notice from the interview facts.
    Standalone -- called on demand (e.g. a 'Generate notice' button), not in the graph."""
    messages = [
        {"role": "system", "content":
         f"You are a legal drafting assistant. Write a formal, respectful legal "
         f"DEMAND NOTICE in {language} based only on the facts provided. Include: "
         f"date placeholder [DATE], sender line 'From: [Your Name, Address]', "
         f"recipient line 'To: [Recipient Name, Address]', a subject line, numbered "
         f"factual paragraphs, a clear demand with a 15-day deadline, and a closing. "
         f"Use [SQUARE BRACKETS] for any detail you don't have. Do not cite specific "
         f"law sections unless they appear in the facts. Output only the notice text."},
        {"role": "user", "content":
         f"Facts: {case_facts.get('key_facts', '')}. "
         f"Issue type: {case_facts.get('issue_type', '')}."},
    ]
    _, clean = gemma_chat(messages, tools=None, max_new_tokens=500)
    return clean.strip()

# Test with real facts from the last run if available; skip cleanly if not.
if "final_state" in globals():
    print(draft_notice(final_state["case_facts"], final_state.get("language", "English")))
else:
    print("(run the pipeline test cell first, then re-run this to draft a notice)")

(run the pipeline test cell first, then re-run this to draft a notice)


In [21]:
def run_interactively(question: str, scripted_answers=None, max_rounds: int = 4):
    for round_num in range(1, max_rounds + 1):
        state: AgentState = {
            "question": question,
            "language": "English",
            "case_facts": {}, "advisor_answer": "", "retrieved_chunks": [],
            "draft": "", "eval_result": {}, "retry_count": 0,
            "needs_more_info": False, "clarifying_questions": [],
        }
        final_state = app.invoke(state)
        print(f"--- interview round {round_num} ---")
        print("case_facts:", final_state["case_facts"])

        if not final_state["needs_more_info"]:
            return final_state

        print("\nThe Interviewer needs a bit more before it can proceed:")
        answers = []
        for q in final_state["clarifying_questions"]:
            if scripted_answers is not None:
                ans = scripted_answers.pop(0) if scripted_answers else "not sure"
                print(q, "->", ans)
            else:
                ans = input(q + "  ")
            answers.append(f"{q} -> {ans}")
        question = question + "\n" + "\n".join(answers)
        print()

    print(f"(hit {max_rounds}-round interview limit, proceeding with what we have)")
    return final_state

In [24]:
final_state = run_interactively(
    "ante veettudamasthan anyaayamaaya securitti depposittu eduthu",
    # scripted_answers=["वह बिना कारण पूरी राशि रोक रहा है",
    #                   "किरायेदारी जनवरी 2026 में खत्म हुई",
    #                   "गाज़ियाबाद, उत्तर प्रदेश"],
)

if final_state.get("needs_more_info"):
    print("Still missing info after max rounds:", final_state["clarifying_questions"])
else:
    print("=== CASE CARD ===")
    print(final_state["draft"])
    print("\n=== EVAL RESULT ===")
    print(final_state["eval_result"])

--- interview round 1 ---
case_facts: {'language': 'Malayalam', 'ready': False, 'issue_type': 'Property Dispute/Theft', 'key_facts': 'The user claims someone took their house/property and installed security.', 'missing_info': ['എന്താണ് സംഭവിച്ചത്? (What exactly happened?)', 'എപ്പോഴാണ് ഇത് സംഭവിച്ചത്? (When did this happen?)', 'ആരാണ് ഇത് ചെയ്തത്? (Who did this do?)']}

The Interviewer needs a bit more before it can proceed:


എന്താണ് സംഭവിച്ചത്? (What exactly happened?)   rep


KeyboardInterrupt: Interrupted by user